<a href="https://colab.research.google.com/github/naresh-FD/react-intelligent-test-generator/blob/main/react_test_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies (takes ~3 min)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets
print("✅ All dependencies installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 19.7 MB

In [ ]:
# Upload dataset.jsonl from your machine
from google.colab import files
print("📁 Select your dataset.jsonl file...")
uploaded = files.upload()
print(f"✅ Uploaded: {list(uploaded.keys())}")

📁 Select your dataset.jsonl file...


Saving dataset.jsonl to dataset (1).jsonl
✅ Uploaded: ['dataset (1).jsonl']


In [ ]:
from unsloth import FastLanguageModel
import torch, json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ Model loaded with LoRA adapters")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Model loaded with LoRA adapters
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
from datasets import Dataset

# Load your dataset (uploaded as "dataset (1).jsonl")
import os
# Find the actual dataset file
for fname in ["dataset.jsonl", "dataset (1).jsonl"]:
    if os.path.exists(fname):
        dataset_file = fname
        break

print(f"Using dataset file: {dataset_file}")

examples = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

print(f"📊 Loaded {len(examples)} training examples")

# Format for Qwen chat template
def format_example(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )}

dataset = Dataset.from_list([format_example(ex) for ex in examples])
print(f"✅ Dataset ready: {len(dataset)} formatted examples")
print(f"\n📝 Preview:\n{dataset[0]['text'][:300]}...")

Using dataset file: dataset.jsonl
📊 Loaded 58 training examples
✅ Dataset ready: 58 formatted examples

📝 Preview:
<|im_start|>system
You are a React testing expert. Given a React/TypeScript source file, generate a comprehensive Jest + React Testing Library test file.

Rules:
- Use Jest 29 + @testing-library/react 16 + @testing-library/user-event 14
- Import test globals from @jest/globals
- Use renderWithProvid...


In [ ]:
# Fix trl version to match unsloth 2026
import subprocess
subprocess.run(["pip", "install", "-q", "--upgrade", "trl"], check=True)
print("✅ trl upgraded")

from trl import SFTTrainer
from transformers import TrainingArguments
import importlib, trl
importlib.reload(trl)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        save_strategy="epoch",
        seed=42,
        gradient_checkpointing=True,
    ),
)

print("🚀 Training started... (30-60 min)")
stats = trainer.train()
print(f"\n✅ DONE! Loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s")

✅ trl upgraded


ImportError: cannot import name '_is_package_available' from 'trl.import_utils' (/usr/local/lib/python3.12/dist-packages/trl/import_utils.py)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        save_strategy="epoch",
        seed=42,
        gradient_checkpointing=True,
    ),
)

print("🚀 Training started... (30-60 min)")
stats = trainer.train()
print(f"\n✅ DONE! Loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'processing_class'

In [ ]:
import sys

# Force-reload trl after upgrade
for key in list(sys.modules.keys()):
    if 'trl' in key:
        del sys.modules[key]

from trl import SFTTrainer
from transformers import TrainingArguments
import trl
print(f"trl version: {trl.__version__}")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        save_strategy="epoch",
        seed=42,
        gradient_checkpointing=True,
    ),
)

print("🚀 Training started... (30-60 min)")
stats = trainer.train()
print(f"\n✅ DONE! Loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
from google.colab import files
files.download("/content/testgen-coder-finetuned_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>